# Real-Life Gemini AI Agent with Long-Term Memory

This notebook builds a practical AI agent using **Gemini + LangChain + SQLite**.

### Architecture

```text
User → LangChain Agent → Gemini
                 ↓
          Memory Tools
                 ↓
              SQLite
                 ↓
        Old conversation search
```

The important idea is that we **do not send the entire old conversation to Gemini**. We keep recent messages in context and retrieve relevant old messages from SQLite only when needed.

## 1. Install packages

Run this cell once.

In [26]:
%pip install -U langchain langchain-core langchain-google-genai python-dotenv

In [27]:
%pip install -U python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [28]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
E:\AI-ML\.venv310\Scripts\python.exe

Python version:
3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [29]:
import site

print(site.getsitepackages())

['E:\\AI-ML\\.venv310', 'E:\\AI-ML\\.venv310\\lib\\site-packages']


In [30]:
import sys

!{sys.executable} -m pip install -U python-dotenv

In [31]:
from dotenv import load_dotenv

print("python-dotenv imported successfully!")

python-dotenv imported successfully!


## 2. Add your Gemini API key

For a real project, keep the key in a `.env` file rather than putting it directly into code.

Create a `.env` file in the same folder as this notebook:

```text
GOOGLE_API_KEY=YOUR_GEMINI_API_KEY
```

In [47]:
import os
from getpass import getpass

GOOGLE_API_KEY = getpass("enter your api ")


if not GOOGLE_API_KEY:
    raise RuntimeError("Gemini API key was not entered.")

print("Gemini API key loaded successfully.")

enter your api  ········


Gemini API key loaded successfully.


In [48]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
)

response = llm.invoke("Say hello in one short sentence.")

print(response.content)

E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Hello, how can I help you today?', 'extras': {'signature': 'El4KXAERTTIPU48j8NcxtukmuNnKiuv02JOwjZpm4LHsf5w4jnjYAqVkva3Gq6xUD15EPylRIg8LKWkuhHHQXdB0ZjPlL3ptuB0c8kdhfBSpFkgpySlFgdZh4gX5wG6t'}}]


In [49]:
import os
from pathlib import Path
from dotenv import load_dotenv

print("Current folder:")
print(Path.cwd())

print("\n.env exists:")
print(Path(".env").exists())

print("\nAPI key loaded:")
print(bool(GOOGLE_API_KEY))

if GOOGLE_API_KEY:
    print("Key length:", len(GOOGLE_API_KEY))

Current folder:
E:\AI-ML

.env exists:
True

API key loaded:
True
Key length: 53


## 3. Imports and configuration

In [50]:
import sqlite3
from datetime import datetime
from pathlib import Path
from typing import Optional

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

DB_PATH = Path("agent_memory.db")
SESSION_ID = "main_user_session"
MODEL_NAME = "gemini-3.5-flash-lite"
RECENT_MESSAGE_LIMIT = 12
MEMORY_SEARCH_LIMIT = 8

print("Database:", DB_PATH.resolve())
print("Model:", MODEL_NAME)

Database: E:\AI-ML\agent_memory.db
Model: gemini-3.5-flash-lite


## 4. SQLite long-term memory

Every conversation is saved locally. The database can contain a very large number of messages without putting all of them into Gemini's context.

In [51]:
class MemoryDB:
    """SQLite-based long-term conversation memory."""

    def __init__(self, db_path: Path):
        self.db_path = str(db_path)
        self._create_tables()

    def connect(self):
        connection = sqlite3.connect(self.db_path)
        connection.row_factory = sqlite3.Row
        return connection

    def _create_tables(self):
        with self.connect() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS messages (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL,
                    content TEXT NOT NULL,
                    created_at TEXT NOT NULL
                )
            """)

            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_messages_session
                ON messages(session_id)
            """)

            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_messages_created
                ON messages(created_at)
            """)

            conn.execute("""
                CREATE TABLE IF NOT EXISTS summaries (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    summary TEXT NOT NULL,
                    created_at TEXT NOT NULL
                )
            """)
            conn.commit()

    def add_message(self, session_id, role, content):
        with self.connect() as conn:
            conn.execute(
                """INSERT INTO messages
                (session_id, role, content, created_at)
                VALUES (?, ?, ?, ?)""",
                (session_id, role, content,
                 datetime.now().isoformat(timespec="seconds"))
            )
            conn.commit()

    def get_recent_messages(self, session_id, limit=12):
        with self.connect() as conn:
            rows = conn.execute(
                """SELECT role, content, created_at
                FROM messages
                WHERE session_id = ?
                ORDER BY id DESC LIMIT ?""",
                (session_id, limit)
            ).fetchall()
        return list(reversed(rows))

    def search_messages(self, query, session_id=None, limit=8):
        terms = [
            word.strip().lower()
            for word in query.split()
            if len(word.strip()) >= 3
        ]
        if not terms:
            return []

        clauses = []
        parameters = []
        for term in terms[:8]:
            clauses.append("LOWER(content) LIKE ?")
            parameters.append(f"%{term}%")

        where = " OR ".join(clauses)

        if session_id:
            sql = f"""SELECT id, role, content, created_at
                     FROM messages
                     WHERE session_id = ? AND ({where})
                     ORDER BY id DESC LIMIT ?"""
            parameters = [session_id] + parameters + [limit]
        else:
            sql = f"""SELECT id, session_id, role, content, created_at
                     FROM messages
                     WHERE ({where})
                     ORDER BY id DESC LIMIT ?"""
            parameters += [limit]

        with self.connect() as conn:
            return conn.execute(sql, parameters).fetchall()

    def get_message_count(self, session_id=None):
        with self.connect() as conn:
            if session_id:
                row = conn.execute(
                    "SELECT COUNT(*) AS count FROM messages WHERE session_id = ?",
                    (session_id,)
                ).fetchone()
            else:
                row = conn.execute(
                    "SELECT COUNT(*) AS count FROM messages"
                ).fetchone()
        return row["count"]

    def save_summary(self, session_id, summary):
        with self.connect() as conn:
            conn.execute(
                """INSERT INTO summaries
                (session_id, summary, created_at)
                VALUES (?, ?, ?)""",
                (session_id, summary,
                 datetime.now().isoformat(timespec="seconds"))
            )
            conn.commit()

memory = MemoryDB(DB_PATH)
print("SQLite memory is ready.")

SQLite memory is ready.


## 5. Create Gemini

In [53]:
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
)

print("Gemini model initialized.")

Gemini model initialized.


## 6. Memory tools

The agent can call these tools when it needs information from the long-term memory.

In [52]:
@tool
def search_old_messages(query: str) -> str:
    """Search older conversation messages stored in SQLite."""
    rows = memory.search_messages(
        query=query,
        session_id=SESSION_ID,
        limit=MEMORY_SEARCH_LIMIT,
    )

    if not rows:
        return "No relevant older messages were found in memory."

    return "\n".join(
        f"[{row['created_at']}] {row['role']}: {row['content']}"
        for row in rows
    )


@tool
def get_memory_statistics() -> str:
    """Return the number of stored messages."""
    return f"This session has {memory.get_message_count(SESSION_ID)} stored messages."


@tool
def get_recent_memory() -> str:
    """Return the latest stored messages."""
    rows = memory.get_recent_messages(
        SESSION_ID,
        limit=RECENT_MESSAGE_LIMIT,
    )

    if not rows:
        return "No messages are stored yet."

    return "\n".join(
        f"[{row['created_at']}] {row['role']}: {row['content']}"
        for row in rows
    )

## 7. Create the LangChain agent

In [54]:
SYSTEM_PROMPT = """
You are a helpful real-life personal AI agent.

Recent messages are supplied directly in your context.
Older conversations are stored in SQLite and can be searched
with the search_old_messages tool.

Rules:
- Use search_old_messages when the user asks about older conversations.
- Do not claim to remember information unless it is available in context or memory.
- Do not retrieve the entire database.
- Retrieve only information relevant to the current question.
- If memories conflict, explain the uncertainty.
- Never expose API keys or secrets.
"""

agent = create_agent(
    model=llm,
    tools=[
        search_old_messages,
        get_memory_statistics,
        get_recent_memory,
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent created successfully.")

Agent created successfully.


## 8. Build the short-term context

Only the latest few messages are sent directly to the agent. This prevents an ever-growing prompt.

In [55]:
def build_messages(user_message: str):
    recent = memory.get_recent_messages(
        SESSION_ID,
        limit=RECENT_MESSAGE_LIMIT,
    )

    messages = [
        {"role": row["role"], "content": row["content"]}
        for row in recent
    ]

    messages.append({
        "role": "user",
        "content": user_message,
    })

    return messages

## 9. Main `agent.invoke()` function

This is the function you can later connect to Streamlit, FastAPI, a web application, or another project.

In [56]:
def ask_agent(user_message: str) -> str:
    messages = build_messages(user_message)

    result = agent.invoke({"messages": messages})
    final_message = result["messages"][-1]

    answer = final_message.content
    if not isinstance(answer, str):
        answer = str(answer)

    memory.add_message(SESSION_ID, "user", user_message)
    memory.add_message(SESSION_ID, "assistant", answer)

    return answer

## 10. Test the agent

Start with a fact that the agent can store.

In [57]:
print(ask_agent("My current project is a personal AI agent with Gemini, LangChain, and SQLite."))

E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "That sounds like a fantastic and practical stack! Building your own personal AI agent gives you total control over your data, privacy, and how it behaves. \n\nUsing **Gemini** gives you a powerful LLM with a great context window, **LangChain** helps structure the agent's workflows, memory, and tool usage, and **SQLite** is a lightweight, zero-config way to persist memory, history, or agent state locally.\n\nHow are you approaching the design right now? Are you working on setting up the SQLite memory backend, connecting custom tools, or something else? Let me know how I can help!", 'extras': {'signature': 'El4KXAERTTIPlrUEfGmezj1LFV9bUJQPaThPXmE1zfBYMEKX1W5/wu7C0lw5faJoGDrs7fvB63HgjJGfU4BVUn5QwWS75I9BFbBPuiOKg4hJGcquqAU1LYG7k8xcEF/t'}}]


Now ask about it later. The recent context may contain it; if it becomes old enough, the agent can use the memory search tool.

In [58]:
print(ask_agent("What project am I building?"))

E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'You are building a personal AI agent using Gemini, LangChain, and SQLite.', 'extras': {'signature': 'El4KXAERTTIPHK4QuhJn/t2q7tFa8zrLrycj9RBNUbM9IcXwDFyW4t6t1jqtkE95Li0FJ9EMgAo3GTUlV6GvjOxWESTZpHfYstML6xjx1zgeUfdOgsb+teeM96h4GtBG'}}]


## 11. Directly search old memory

This is useful for debugging the memory system.

In [59]:
results = memory.search_messages(
    "Gemini LangChain SQLite",
    session_id=SESSION_ID,
    limit=8,
)

for row in results:
    print(f"[{row['created_at']}] {row['role']}: {row['content']}")

[2026-08-26T14:46:26] assistant: [{'type': 'text', 'text': 'You are building a personal AI agent using Gemini, LangChain, and SQLite.', 'extras': {'signature': 'El4KXAERTTIPHK4QuhJn/t2q7tFa8zrLrycj9RBNUbM9IcXwDFyW4t6t1jqtkE95Li0FJ9EMgAo3GTUlV6GvjOxWESTZpHfYstML6xjx1zgeUfdOgsb+teeM96h4GtBG'}}]
[2026-08-26T14:46:23] assistant: [{'type': 'text', 'text': "That sounds like a fantastic and practical stack! Building your own personal AI agent gives you total control over your data, privacy, and how it behaves. \n\nUsing **Gemini** gives you a powerful LLM with a great context window, **LangChain** helps structure the agent's workflows, memory, and tool usage, and **SQLite** is a lightweight, zero-config way to persist memory, history, or agent state locally.\n\nHow are you approaching the design right now? Are you working on setting up the SQLite memory backend, connecting custom tools, or something else? Let me know how I can help!", 'extras': {'signature': 'El4KXAERTTIPlrUEfGmezj1LFV9bUJQPa

## 12. See recent history

In [60]:
recent = memory.get_recent_messages(SESSION_ID, RECENT_MESSAGE_LIMIT)

for row in recent:
    print(f"{row['role'].upper()}: {row['content']}")

USER: My current project is a personal AI agent with Gemini, LangChain, and SQLite.
ASSISTANT: [{'type': 'text', 'text': "That sounds like a fantastic and practical stack! Building your own personal AI agent gives you total control over your data, privacy, and how it behaves. \n\nUsing **Gemini** gives you a powerful LLM with a great context window, **LangChain** helps structure the agent's workflows, memory, and tool usage, and **SQLite** is a lightweight, zero-config way to persist memory, history, or agent state locally.\n\nHow are you approaching the design right now? Are you working on setting up the SQLite memory backend, connecting custom tools, or something else? Let me know how I can help!", 'extras': {'signature': 'El4KXAERTTIPlrUEfGmezj1LFV9bUJQPaThPXmE1zfBYMEKX1W5/wu7C0lw5faJoGDrs7fvB63HgjJGfU4BVUn5QwWS75I9BFbBPuiOKg4hJGcquqAU1LYG7k8xcEF/t'}}]
USER: What project am I building?
ASSISTANT: [{'type': 'text', 'text': 'You are building a personal AI agent using Gemini, LangChain

## 13. Import an old text file

If you have an old `.txt` conversation export, you can import it into SQLite. For JSON/JSONL/CSV exports, create a parser matching the exact export structure.

In [61]:
def import_text_file(file_path: str, session_id: str = SESSION_ID):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(path)

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    imported = 0
    for line in lines:
        line = line.strip()
        if not line:
            continue
        memory.add_message(session_id, "user", line)
        imported += 1

    return imported

# Example:
# imported = import_text_file("old_messages.txt")
# print("Imported:", imported)

## 14. Interactive chat

Run this cell to turn the notebook into a simple chatbot.

Commands:
- `/history` — recent messages
- `/search something` — search old memory
- `/stats` — message count
- `/exit` — stop chat

In [ ]:
while True:
    user_input = input("\nYou: ").strip()

    if not user_input:
        continue

    if user_input == "/exit":
        print("Chat ended.")
        break

    if user_input == "/history":
        rows = memory.get_recent_messages(SESSION_ID, RECENT_MESSAGE_LIMIT)
        for row in rows:
            print(f"{row['role'].upper()}: {row['content']}")
        continue

    if user_input.startswith("/search "):
        query = user_input[len("/search "):].strip()
        rows = memory.search_messages(query, SESSION_ID, MEMORY_SEARCH_LIMIT)
        if not rows:
            print("No matching memories found.")
        else:
            for row in rows:
                print(f"[{row['created_at']}] {row['role']}: {row['content']}")
        continue

    if user_input == "/stats":
        print("Stored messages:", memory.get_message_count(SESSION_ID))
        continue

    try:
        print("\nAgent:", ask_agent(user_input))
    except Exception as e:
        print("Error:", e)


You:  what project i am builiding


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Agent: [{'type': 'text', 'text': 'You are building a personal AI agent using Gemini, LangChain, and SQLite.', 'extras': {'signature': 'El4KXAERTTIPBUkkLe5CvQ13fe1LMyAiqDCfVfWpNQSkqbbcBrbVJMhSjDw3hn7E31Euo3rjmsxsFT6WWxJkqpe0btn+chOT1MixSMGF56rJo8g5xczfYoJVcShHHi45'}}]


## 15. Next upgrade for huge message archives

For a genuinely large archive, such as 100,000+ messages, upgrade the retrieval layer:

```text
Old messages
     ↓
SQLite / FTS5 ─────── keyword search
     +
Embeddings ────────── semantic search
     ↓
Hybrid retrieval
     ↓
Top relevant memories
     ↓
Gemini Agent
```

The current notebook deliberately starts with SQLite keyword search because it is easier to understand and debug. The same `search_old_messages()` tool can later be replaced with semantic/hybrid retrieval without redesigning the entire agent.